# SymboLM (v1.1) — Kaggle Stage 1 SFT Resume (Steps 200 → 675)
**Author:** Sanish Gyawali  
**Model:** `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`  
**Hardware:** NVIDIA Tesla T4 / P100 (16GB VRAM) in native FP16 (Unquantized LoRA)

This notebook automatically downloads `checkpoint-200` from Google Drive and resumes SFT training without quantization mismatch, ensuring 100% numerical consistency with the TPU run.

In [ ]:
# 1. GPU Verification
!nvidia-smi

In [ ]:
# 2. Clone SymboLM and install dependencies
%cd /kaggle/working
!rm -rf symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git
%cd /kaggle/working/symboLM
# Remove Kaggle's incompatible torchao package to prevent PEFT collision
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt
!pip install -q gdown

In [ ]:
# 3. Download & Stage Checkpoint-200 from Google Drive
import os
import shutil
from pathlib import Path
import gdown

dest_ckpt = Path("/kaggle/working/symboLM/checkpoints/sft_adapter/checkpoint-200")
dest_ckpt.mkdir(parents=True, exist_ok=True)

gdrive_url = "https://drive.google.com/drive/folders/1aEJXXog1A9ak3zf4hHqc8bbfy_iFi793"
print(f"Downloading checkpoint from Google Drive: {gdrive_url}...")
gdown.download_folder(gdrive_url, output="/kaggle/working/downloaded_ckpt", quiet=False, use_cookies=False)

# Locate adapter files and stage into destination
ckpt_files = list(Path("/kaggle/working/downloaded_ckpt").glob("**/adapter_model.safetensors"))
if ckpt_files:
    source_dir = ckpt_files[0].parent
    print(f"Staging files from {source_dir} to {dest_ckpt}...")
    for f in source_dir.iterdir():
        if f.is_file():
            shutil.copy2(f, dest_ckpt / f.name)
        elif f.is_dir():
            shutil.copytree(f, dest_ckpt / f.name, dirs_exist_ok=True)
    print(f"✓ Checkpoint-200 staged! Files: {[f.name for f in dest_ckpt.iterdir()]}")
else:
    print("Warning: adapter_model.safetensors not found in downloaded folder.")

In [ ]:
# 4. Resume SFT Training (Steps 200 -> 675)
# --no_quant uses native 16-bit LoRA with adamw_torch matching Checkpoint-200
!python -m training.sft_train \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --data_dir ./data/symbolic_dataset \
    --output_dir ./checkpoints/sft_adapter \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 8 \
    --lr 0.0002 \
    --no_quant \
    --resume_from_checkpoint ./checkpoints/sft_adapter/checkpoint-200

In [ ]:
# 5. Test Inference with Trained Adapter
!python -m inference.generate \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --adapter_path ./checkpoints/sft_adapter \
    --prompt "A train departs with 100 passengers. At the first stop, 20 leave and 35 enter. How many passengers now?"

In [ ]:
# 6. Compress Final Trained Adapter for Easy Download from Kaggle Output
!tar -czvf /kaggle/working/symboLM_sft_final_adapter.tar.gz -C /kaggle/working/symboLM/checkpoints/sft_adapter .
print("Done! Output is saved to /kaggle/working/symboLM_sft_final_adapter.tar.gz")